In [1]:
import gc

gc.collect()

2064

# elevator-rmnd-lstm

Use the simulated dataset generated by `dataset.py` to train
regression models, in order to predict the remaining useful life (RUL)
of lifts.

**This approach uses an LSTM.**

In [2]:
from datetime import datetime
import dataset
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import joblib as jl
import seaborn as sns
import os

sns.set_theme(context="notebook", style="ticks")

MODEL_DIR = os.path.join(os.getcwd(), "models")

## preprocessing
* Import the simulated dataset
* ~~Convert lift ids to integers~~ [already handled by `dataset.py`]
* Compute instantaneous time derivatives of each sensor metric _[working on it]_
* ~~Compute RUL in h~~ [already handled by `dataset.py`]
* Split data into training/testing sets
* Scale data with a `Scaler`

In [3]:
df_full = pd.read_pickle("predictive_maintenance_lifts.pkl")
df_full.head(10)

,timestamp,lift_id,lift_model,lift_age_hours,ARM_DIST_mm,DOOR_DIST_mm,FLOOR_DIST_mm,ROPE_MFL_mV,BEARING_TEMP_C,maintenance_done,RUL_hrs
0,2023-01-01 00:00:00,1,Otis Gen2,15000,0.506,2.101,0.116,9.737,34.429,0,996.0
1,2023-01-01 12:00:00,1,Otis Gen2,15012,0.486,1.811,0.288,10.476,33.628,0,984.0
2,2023-01-02 00:00:00,1,Otis Gen2,15024,0.500,2.070,0.056,9.066,34.675,0,972.0
3,2023-01-02 12:00:00,1,Otis Gen2,15036,0.525,2.472,0.373,10.265,34.941,0,960.0
4,2023-01-03 00:00:00,1,Otis Gen2,15048,0.506,2.383,0.451,9.954,34.587,0,948.0
5,2023-01-03 12:00:00,1,Otis Gen2,15060,0.519,2.307,0.490,10.225,36.151,0,936.0
6,2023-01-04 00:00:00,1,Otis Gen2,15072,0.501,2.541,1.145,11.004,36.132,0,924.0
7,2023-01-04 12:00:00,1,Otis Gen2,15084,0.549,2.403,0.371,10.862,35.528,0,912.0
8,2023-01-05 00:00:00,1,Otis Gen2,15096,0.509,2.488,0.559,10.610,36.800,0,900.0
9,2023-01-05 12:00:00,1,Otis Gen2,15108,0.558,2.738,1.035,10.507,35.747,0,888.0


Compute the time derivatives for each sensor row.

In [4]:
sensor_cols = [
    "ARM_DIST_mm",
    "DOOR_DIST_mm",
    "FLOOR_DIST_mm",
    "ROPE_MFL_mV",
    "BEARING_TEMP_C",
]
sensor_cols_old = sensor_cols.copy()
time_derivs = {}
for col in sensor_cols_old:
    colname_new = col + "_per_hr"
    sensor_cols.append(colname_new)
    time_derivs[colname_new] = np.round(
        df_full[col].copy().diff().fillna(0).div(dataset.HOURS_PER_STEP), 4
    )

In [5]:
df_time_derivs = pd.DataFrame(time_derivs)
print(df_time_derivs.head(10))

   ARM_DIST_mm_per_hr  DOOR_DIST_mm_per_hr  FLOOR_DIST_mm_per_hr  \
0              0.0000               0.0000                0.0000   
1             -0.0017              -0.0242                0.0143   
2              0.0012               0.0216               -0.0193   
3              0.0021               0.0335                0.0264   
4             -0.0016              -0.0074                0.0065   
5              0.0011              -0.0063                0.0032   
6             -0.0015               0.0195                0.0546   
7              0.0040              -0.0115               -0.0645   
8             -0.0033               0.0071                0.0157   
9              0.0041               0.0208                0.0397   

   ROPE_MFL_mV_per_hr  BEARING_TEMP_C_per_hr  
0              0.0000                 0.0000  
1              0.0616                -0.0668  
2             -0.1175                 0.0872  
3              0.0999                 0.0222  
4             -0

In [6]:
df_full = df_full.join(df_time_derivs, how="right")
df_full.head(10)

,timestamp,lift_id,lift_model,lift_age_hours,ARM_DIST_mm,DOOR_DIST_mm,FLOOR_DIST_mm,ROPE_MFL_mV,BEARING_TEMP_C,maintenance_done,RUL_hrs,ARM_DIST_mm_per_hr,DOOR_DIST_mm_per_hr,FLOOR_DIST_mm_per_hr,ROPE_MFL_mV_per_hr,BEARING_TEMP_C_per_hr
0,2023-01-01 00:00:00,1,Otis Gen2,15000,0.506,2.101,0.116,9.737,34.429,0,996.0,0.0000,0.0000,0.0000,0.0000,0.0000
1,2023-01-01 12:00:00,1,Otis Gen2,15012,0.486,1.811,0.288,10.476,33.628,0,984.0,-0.0017,-0.0242,0.0143,0.0616,-0.0668
2,2023-01-02 00:00:00,1,Otis Gen2,15024,0.500,2.070,0.056,9.066,34.675,0,972.0,0.0012,0.0216,-0.0193,-0.1175,0.0872
3,2023-01-02 12:00:00,1,Otis Gen2,15036,0.525,2.472,0.373,10.265,34.941,0,960.0,0.0021,0.0335,0.0264,0.0999,0.0222
4,2023-01-03 00:00:00,1,Otis Gen2,15048,0.506,2.383,0.451,9.954,34.587,0,948.0,-0.0016,-0.0074,0.0065,-0.0259,-0.0295
5,2023-01-03 12:00:00,1,Otis Gen2,15060,0.519,2.307,0.490,10.225,36.151,0,936.0,0.0011,-0.0063,0.0032,0.0226,0.1303
6,2023-01-04 00:00:00,1,Otis Gen2,15072,0.501,2.541,1.145,11.004,36.132,0,924.0,-0.0015,0.0195,0.0546,0.0649,-0.0016
7,2023-01-04 12:00:00,1,Otis Gen2,15084,0.549,2.403,0.371,10.862,35.528,0,912.0,0.0040,-0.0115,-0.0645,-0.0118,-0.0503
8,2023-01-05 00:00:00,1,Otis Gen2,15096,0.509,2.488,0.559,10.610,36.800,0,900.0,-0.0033,0.0071,0.0157,-0.0210,0.1060
9,2023-01-05 12:00:00,1,Otis Gen2,15108,0.558,2.738,1.035,10.507,35.747,0,888.0,0.0041,0.0208,0.0397,-0.0086,-0.0877


In [7]:
# Remove unnecessary columns
df_full = df_full.drop(columns=["timestamp", "lift_model", "maintenance_done"])

Since there exist rows that have `np.inf` as their predicted RUL, extract those rows for experimentation later on.
Retain the remaining roles (with valid RULs) for training and testing.

In [8]:
df_experimental = df_full[df_full["RUL_hrs"] == np.inf]
df_useful = df_full[df_full["RUL_hrs"] != np.inf]

# Check to see the experimental/useful split was done correctly
assert all(
    df_experimental["RUL_hrs"] == np.inf
), "All records in experimental set should have RUL_hrs == np.inf"
assert all(
    df_useful["RUL_hrs"] != np.inf
), "All records in useful set should have valid RUL_hrs"

df_useful.head(40)

,lift_id,lift_age_hours,ARM_DIST_mm,DOOR_DIST_mm,FLOOR_DIST_mm,ROPE_MFL_mV,BEARING_TEMP_C,RUL_hrs,ARM_DIST_mm_per_hr,DOOR_DIST_mm_per_hr,FLOOR_DIST_mm_per_hr,ROPE_MFL_mV_per_hr,BEARING_TEMP_C_per_hr
0,1,15000,0.506,2.101,0.116,9.737,34.429,996.0,0.0000,0.0000,0.0000,0.0000,0.0000
1,1,15012,0.486,1.811,0.288,10.476,33.628,984.0,-0.0017,-0.0242,0.0143,0.0616,-0.0668
2,1,15024,0.500,2.070,0.056,9.066,34.675,972.0,0.0012,0.0216,-0.0193,-0.1175,0.0872
3,1,15036,0.525,2.472,0.373,10.265,34.941,960.0,0.0021,0.0335,0.0264,0.0999,0.0222
4,1,15048,0.506,2.383,0.451,9.954,34.587,948.0,-0.0016,-0.0074,0.0065,-0.0259,-0.0295
5,1,15060,0.519,2.307,0.490,10.225,36.151,936.0,0.0011,-0.0063,0.0032,0.0226,0.1303
6,1,15072,0.501,2.541,1.145,11.004,36.132,924.0,-0.0015,0.0195,0.0546,0.0649,-0.0016
7,1,15084,0.549,2.403,0.371,10.862,35.528,912.0,0.0040,-0.0115,-0.0645,-0.0118,-0.0503
8,1,15096,0.509,2.488,0.559,10.610,36.800,900.0,-0.0033,0.0071,0.0157,-0.0210,0.1060
9,1,15108,0.558,2.738,1.035,10.507,35.747,888.0,0.0041,0.0208,0.0397,-0.0086,-0.0877


In [9]:
sensor_cols

['ARM_DIST_mm',
 'DOOR_DIST_mm',
 'FLOOR_DIST_mm',
 'ROPE_MFL_mV',
 'BEARING_TEMP_C',
 'ARM_DIST_mm_per_hr',
 'DOOR_DIST_mm_per_hr',
 'FLOOR_DIST_mm_per_hr',
 'ROPE_MFL_mV_per_hr',
 'BEARING_TEMP_C_per_hr']

## sequences for LSTM training

In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

First create train/val/test splits in the ratio 70/15/15.

In [ ]:
# Split df_useful into 70/15/15 train/val/test sets
train_val, test_df = train_test_split(df_useful, test_size=0.15, random_state=42)
train_df, val_df = train_test_split(
    train_val, test_size=0.1765, random_state=42
)  # 0.1765*0.85 ≈ 0.15
train_df.head(20)

,lift_id,lift_age_hours,ARM_DIST_mm,DOOR_DIST_mm,FLOOR_DIST_mm,ROPE_MFL_mV,BEARING_TEMP_C,RUL_hrs,ARM_DIST_mm_per_hr,DOOR_DIST_mm_per_hr,FLOOR_DIST_mm_per_hr,ROPE_MFL_mV_per_hr,BEARING_TEMP_C_per_hr
30014,2,152168,0.598,4.449,1.336,12.757,45.014,588.0,-0.0012,0.0218,0.0062,0.1087,0.0693
35940,2,223280,0.627,3.464,0.638,11.156,40.602,840.0,0.0008,0.0019,-0.0143,0.0475,0.0781
8356,1,115272,0.676,4.400,8.037,15.091,42.819,876.0,0.0016,-0.0011,-0.0092,-0.0035,-0.0511
29802,2,149624,0.868,4.597,2.310,13.410,54.861,684.0,-0.0015,0.0011,-0.0148,0.0376,0.1569
1343,1,31116,0.542,2.276,0.277,10.288,36.372,1008.0,-0.0003,0.0065,0.0043,-0.0532,-0.0217
13809,1,180708,0.992,5.172,2.847,14.825,47.283,924.0,-0.0019,-0.0227,0.0177,-0.0672,0.0342
12701,1,167412,0.783,4.689,5.013,11.529,58.387,900.0,0.0010,0.0222,-0.0332,-0.0506,0.1965
42533,3,38896,0.778,5.385,4.623,14.674,58.473,456.0,-0.0017,-0.0097,0.0201,0.0139,-0.1866
1487,1,32844,0.911,3.556,3.123,11.750,58.775,612.0,0.0029,0.0073,0.0003,-0.0789,0.2449
12194,1,161328,1.141,5.451,5.440,20.268,77.434,72.0,0.0020,0.0059,0.0434,-0.0053,-0.0168


Then scale the data with `StandardScaler`.

In [12]:
scaler = StandardScaler()
train_df[sensor_cols] = scaler.fit_transform(train_df[sensor_cols])
val_df[sensor_cols] = scaler.transform(val_df[sensor_cols])
test_df[sensor_cols] = scaler.transform(test_df[sensor_cols])

Lastly, create sequences for training with an RNN.

In [ ]:
def create_seqs(data, seq_len, feature_cols, target_col="RUL_hrs"):
    X, y = [], []
    features, targets = data[feature_cols].values, data[target_col].values
    for i in range(len(data) - seq_len):
        X.append(features[i : i + seq_len])
        y.append(targets[i + seq_len])
    return np.array(X), np.array(y)


X_train, y_train = create_seqs(train_df, seq_len=50, feature_cols=sensor_cols)
X_val, y_val = create_seqs(val_df, seq_len=50, feature_cols=sensor_cols)
X_test, y_test = create_seqs(test_df, seq_len=50, feature_cols=sensor_cols)

## build an LSTM!
We use `pytorch_lightning` to build an LSTM and then train it on the datasets created.

In [31]:
import torch
import torchmetrics as tm
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# Some important constants
BATCH_SIZE = 32
N_EPOCHS = 25
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
DROPOUT = 0.2
MODEL_DIR = os.path.join(os.getcwd(), "lightning_logs/checkpoints/")

We first create the appropriate datasets and dataloaders, using `TensorDataset` and `DataLoader`.

In [ ]:
# create datasets and dataloaders
train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)
val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

train_dataset, val_loader

(<torch.utils.data.dataset.TensorDataset at 0x2e6e2cc8290>,
 <torch.utils.data.dataloader.DataLoader at 0x2e6e2c3d590>)

In [ ]:
class BasicLSTM(pl.LightningModule):
    """A simple LSTM-based regression model for RUL prediction.
    All subsequent models will use this class as a parent."""
    def __init__(
        self,
        model_dir: str = MODEL_DIR,
        input_size: int = X_train.shape[2],
        output_size: int = 1,
        lr: float = LEARNING_RATE,
        weight_decay: float = WEIGHT_DECAY,
        dropout: float = DROPOUT,
        n_epochs: int = N_EPOCHS,
    ):
        super(BasicLSTM, self).__init__()
        self.save_hyperparameters()
        self.hparams.n_epochs = n_epochs
        self.model_dir = model_dir
        torch.set_float32_matmul_precision("high")

        self.lstm1 = nn.LSTM(
            input_size, 64, num_layers=1, batch_first=True, dropout=self.hparams.dropout
        )
        self.lstm2 = nn.LSTM(64, 32, num_layers=1, batch_first=True, dropout=self.hparams.dropout)
        self.fc = nn.Linear(32, output_size)
        self.criterion = nn.MSELoss()

        # assess a few more metrics
        self.mae = tm.MeanAbsoluteError()
        self.r2 = tm.R2Score()
        self.train_mae = self.mae.clone()
        self.val_mae = self.mae.clone()
        self.test_mae = self.mae.clone()
        self.train_r2 = self.r2.clone()
        self.val_r2 = self.r2.clone()
        self.test_r2 = self.r2.clone()

    def forward(self, x):
        out, _ = self.lstm1(x)
        out = nn.ReLU()(out)
        out, _ = self.lstm2(out)
        out = nn.ReLU()(out)
        out = self.fc(out[:, -1, :])
        return out.squeeze(-1)

    def _shared_step(self, batch):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y.float())
        return loss, y, y_hat

    def training_step(self, batch, batch_idx):
        loss, truths, preds = self._shared_step(batch)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        mae = self.train_mae(preds, truths.float())
        r2 = self.train_r2(preds, truths.float())
        self.log("train_mae", mae, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_r2", r2, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, truths, preds = self._shared_step(batch)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        mae = self.val_mae(preds, truths.float())
        r2 = self.val_r2(preds, truths.float())
        self.log("val_mae", mae, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_r2", r2, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        loss, truths, preds = self._shared_step(batch)
        self.log("test_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        mae = self.test_mae(preds, truths.float())
        r2 = self.test_r2(preds, truths.float())
        self.log("test_mae", mae, on_step=False, on_epoch=True, prog_bar=True)
        self.log("test_r2", r2, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def init_callbacks(self):
        now = datetime.now().strftime("%Y%m%d-%H%M")
        early_stop = pl.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, mode="min", verbose=True,
        )
        checkpoint = pl.callbacks.ModelCheckpoint(
            monitor="val_loss",
            dirpath=f"{self.model_dir}/{now}",
            filename="{epoch:02d}-{val_loss:.4f}",
            save_top_k=1,
            mode="min",
            verbose=True,
        )
        lr_monitor = pl.callbacks.LearningRateMonitor(logging_interval="epoch")
        return [early_stop, checkpoint, lr_monitor]

    def init_trainer(self, max_epochs=self.hparams.n_epochs, callbacks=None):
        return pl.Trainer(
            max_epochs=max_epochs,
            callbacks=callbacks,
            accelerator="auto",
            devices="auto",
            log_every_n_steps=100,
            gradient_clip_val=1.0, 
            deterministic=False,
            precision="16-mixed" if torch.cuda.is_available() else "32",
        )

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=3
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1,
            },
        }

NameError: name 'pl' is not defined

In [ ]:
model = 
    input_size=X_train.shape[2],
    num_layers=2,
    output_size=1, # predict a single value (RUL_hrs)
    dropout=0.2,
    lr=LEARNING_RATE,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores

   | Name      | Type              | Params | Mode  | FLOPs
-----------------------------------------------------------------
0  | lstm1     | LSTM              | 52.7 K | train | 0    
1  | lstm2     | LSTM              | 21.0 K | train | 0    
2  | fc        | Linear            | 33     | train | 0    
3  | criterion | MSELoss           | 0      | train | 0    
4  | mae       | MeanAbsoluteError | 0      | train | 0    
5  | r2        | R2Score           | 0      | train | 0    
6  | train_mae | MeanAbsoluteError | 0      | train | 0    
7  | val_mae   | MeanAbsoluteError | 0      | train | 0    
8  | test_mae  | MeanAbsoluteError | 0      | train | 0    
9  | train_r2  | R2Score           | 0      | train | 0    
10 | val_r2    | R2Score           | 0      | train | 0    
11 | test_r2   | R2Score           | 0      | train | 0    
-----------------------------------------------------------------
73.8 K    Tr

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\darre\OneDrive\Desktop\etps26\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
c:\Users\darre\OneDrive\Desktop\etps26\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.


Finally, test the model.

In [35]:
trainer.test(model, test_loader, ckpt_path="best")
print("\nbest model saved at:", ckpt_callback.best_model_path)
print("best MAE:", ckpt_callback.best_model_score)

Restoring states from the checkpoint path at c:\Users\darre\OneDrive\Desktop\etps26\lightning_logs\version_1\checkpoints\epoch=19-step=13100.ckpt
Loaded model weights from the checkpoint at c:\Users\darre\OneDrive\Desktop\etps26\lightning_logs\version_1\checkpoints\epoch=19-step=13100.ckpt
c:\Users\darre\OneDrive\Desktop\etps26\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss               510510.125
        test_mae             572.1838989257812
         test_r2            -1.4820075035095215
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


NameError: name 'ckpt_callback' is not defined